# 08 Exercises

Part 3 E02 (BatchNorm'u önceki Linear katmana katlama / folding) ve Part 2 E01 (Hiperparametre optimizasyonu ile validation loss < 2.2).


In [ ]:
import random
import torch
import torch.nn.functional as F

# 1. Part 3 E02: BatchNorm'u Linear Katmanın W ve b'sine katlama (Folding)
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3
n_emb = 10
n_hidden = 100

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_emb), generator=g)
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g) * 0.2
b1 = torch.randn(n_hidden, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.5 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.2
bnmean_running = torch.randn((1, n_hidden), generator=g) * 0.1
bnstd_running = torch.rand((1, n_hidden), generator=g) + 0.5
eps = 1e-5

# Test girdisi
X_sample = torch.randint(0, 27, (10, block_size))
emb = C[X_sample].view(-1, n_emb * block_size)

# Standart BatchNorm ileri gecis:
# hpreact_norm = (emb @ W1 + b1 - bnmean_running) / (bnstd_running + eps) * bngain + bnbias
linear_out = emb @ W1 + b1
bn_out = (linear_out - bnmean_running) / (bnstd_running + eps) * bngain + bnbias

# Katlanmis Linear agirlik ve bias:
# (X @ W1 + b1 - mu) * (gamma / sigma) + beta = X @ [W1 * (gamma / sigma)] + [(b1 - mu) * (gamma / sigma) + beta]
scale = bngain / (bnstd_running + eps)
W_folded = W1 * scale
b_folded = (b1 - bnmean_running) * scale + bnbias

folded_out = emb @ W_folded + b_folded

max_diff = (bn_out - folded_out).abs().max().item()
print("BatchNorm vs Katlanmis Linear maksimum cikis farki:", max_diff)
assert max_diff < 1e-5, "Folding dogrulamasi basarisiz!"
print("Dogrulama basarili: BatchNorm parametreleri Linear katmana kusursuzca katlandi.")

# 2. Part 2 E01: Hiperparametreleri ayarlayarak Karpathy 2.2 val loss hedefini gecme
def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1], block_size=3)
Xdev, Ydev = build_dataset(words[n1:n2], block_size=3)

n_emb = 10
n_hidden = 200
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, n_emb), generator=g, requires_grad=True)
W1 = torch.randn((n_emb * 3, n_hidden), generator=g) * (5/3) / ((n_emb * 3)**0.5)
W1.requires_grad = True
W2 = torch.randn((n_hidden, 27), generator=g) * 0.01
W2.requires_grad = True
b2 = torch.randn(27, generator=g) * 0
b2.requires_grad = True
bngain = torch.ones((1, n_hidden), requires_grad=True)
bnbias = torch.zeros((1, n_hidden), requires_grad=True)
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

params = [C, W1, W2, b2, bngain, bnbias]

for i in range(35000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C[Xtr[ix]].view(-1, n_emb * 3)
    hpreact = emb @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)
    hpreact_norm = (hpreact - bnmean) / (bnstd + 1e-5) * bngain + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstd
    h = torch.tanh(hpreact_norm)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    
    for p in params:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 25000 else 0.01
    for p in params:
        p.data += -lr * p.grad

emb_dev = C[Xdev].view(-1, n_emb * 3)
hpreact_dev = emb_dev @ W1
hpreact_dev_norm = (hpreact_dev - bnmean_running) / (bnstd_running + 1e-5) * bngain + bnbias
h_dev = torch.tanh(hpreact_dev_norm)
logits_dev = h_dev @ W2 + b2
val_loss = F.cross_entropy(logits_dev, Ydev).item()
print(f"Validation loss: {val_loss:.4f} (Karpathy 2.2 hedefini gecti mi: {val_loss < 2.2})")
